In [13]:
!git clone https://github.com/saif01234567/CareLink.git /content/CareLink
%cd /content/CareLink
!git checkout --detach 210e8b51ec9dca1b882d1f3036ac36db791d4e37
!git log -1 --oneline

fatal: destination path '/content/CareLink' already exists and is not an empty directory.
/content/CareLink
HEAD is now at 210e8b5 docs: update project progress and next steps
210e8b5 (HEAD, origin/feat/synthetic-threshold-review) docs: update project progress and next steps


In [14]:
import sys
import subprocess

print("Python version:", sys.version)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "-r", "/content/CareLink/ML/synthetic/requirements-baseline.txt"
    ],
    check=True
)

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-r', '/content/CareLink/ML/synthetic/requirements-baseline.txt'], returncode=0)

In [15]:
import subprocess
import sys

for script in ["test_prepare_examples.py", "test_train_baseline.py"]:
    result = subprocess.run(
        [sys.executable, "-B", f"/content/CareLink/ML/synthetic/{script}"],
        capture_output=True,
        text=True
    )
    print(f"\n--- {script} ---")
    print(result.stdout)
    print(result.stderr)
    result.check_returncode()



--- test_prepare_examples.py ---

.....
----------------------------------------------------------------------
Ran 5 tests in 0.001s

OK


--- test_train_baseline.py ---

...
----------------------------------------------------------------------
Ran 3 tests in 0.004s

OK



In [16]:
from pathlib import Path
import subprocess
import sys

demo_folder = Path("/content/CareLink-demo-data")

if demo_folder.exists():
    print("Dataset folder already exists. Run the verification cell below.")
else:
    subprocess.run(
        [
            sys.executable, "-B",
            "/content/CareLink/ML/synthetic/generate_demo.py",
            "--output", str(demo_folder),
            "--seed", "42"
        ],
        check=True
    )

Dataset folder already exists. Run the verification cell below.


In [17]:
from pathlib import Path
import json
import hashlib

folder = Path("/content/CareLink-demo-data")
manifest = json.loads((folder / "manifest.json").read_text())

for filename, expected_hash in manifest["sha256"].items():
    actual_hash = hashlib.sha256((folder / filename).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, f"File check failed: {filename}"

print("All dataset files verified.")
print("Patients:", manifest["patients"])
print("Days per patient:", manifest["days_per_patient"])
print("Dose records:", manifest["dose_events"])
print("Seed:", manifest["seed"])

All dataset files verified.
Patients: 120
Days per patient: 90
Dose records: 16380
Seed: 42


In [18]:
from pathlib import Path
import subprocess
import sys
import tempfile

# A new folder each run prevents overwriting earlier results.
examples_folder = Path(
    tempfile.mkdtemp(prefix="carelink-preparation-", dir="/content")
) / "examples"

result = subprocess.run(
    [
        sys.executable, "-B",
        "/content/CareLink/ML/synthetic/prepare_examples.py",
        "--source", "/content/CareLink-demo-data",
        "--output", str(examples_folder)
    ],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
result.check_returncode()
print("Saved to:", examples_folder)

{
  "is_synthetic": true,
  "source_sha256": "ee3335cfe0f650b562f69f6c4ea085e9fbae16b5fb7170399e54fc8ba4013d5c",
  "feature_allowlist": [
    "scheduled_doses_14d",
    "taken_reports_14d",
    "skipped_reports_14d",
    "unconfirmed_doses_14d",
    "taken_reports_3d",
    "skipped_reports_3d",
    "unconfirmed_doses_3d"
  ],
  "examples": 9120,
  "outcomes": {
    "unknown": 2072,
    "all_reported_taken": 4555,
    "reported_skip": 2493
  },
  "split_outcomes": {
    "train": {
      "unknown": 1542,
      "all_reported_taken": 3107,
      "reported_skip": 1735
    },
    "validation": {
      "all_reported_taken": 720,
      "reported_skip": 361,
      "unknown": 287
    },
    "test": {
      "all_reported_taken": 728,
      "unknown": 243,
      "reported_skip": 397
    }
  },
  "unknown_fraction": 0.22719298245614036,
  "observation_assumption": "Completed synthetic simulation, observed through each label deadline; not for live extracts",
  "warning": "Synthetic-only pipeline dem

In [19]:
from pathlib import Path
import subprocess
import sys
import tempfile

training_folder = Path(
    tempfile.mkdtemp(prefix="carelink-training-", dir="/content")
) / "results"

result = subprocess.run(
    [
        sys.executable, "-B",
        "/content/CareLink/ML/synthetic/train_baseline.py",
        "--source", str(examples_folder / "daily_examples_SYNTHETIC.csv"),
        "--output", str(training_folder)
    ],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
result.check_returncode()
print("Training evidence saved to:", training_folder)

{
  "is_synthetic": true,
  "evaluation_split": "validation",
  "test_evaluated": false,
  "threshold": 0.5,
  "hyperparameter_search": false,
  "training_rows": 4842,
  "validation_rows": 1081,
  "excluded_unknown": {
    "train": 1542,
    "validation": 287
  },
  "training_positive_rate": 0.3583230070218918,
  "validation_positive_rate": 0.333950046253469,
  "feature_allowlist": [
    "scheduled_doses_14d",
    "taken_reports_14d",
    "skipped_reports_14d",
    "unconfirmed_doses_14d",
    "taken_reports_3d",
    "skipped_reports_3d",
    "unconfirmed_doses_3d"
  ],
  "metrics": {
    "majority_baseline": {
      "accuracy": 0.666049953746531,
      "balanced_accuracy": 0.5,
      "precision": 0.0,
      "recall": 0.0,
      "f1": 0.0,
      "roc_auc": 0.5,
      "average_precision": 0.333950046253469,
      "brier_score": 0.333950046253469,
      "confusion_matrix_rows_actual_0_1_columns_predicted_0_1": [
        [
          720,
          0
        ],
        [
          361,
   

In [20]:
import shutil
from google.colab import files

archive = shutil.make_archive(
    "/content/CareLink_Synthetic_Training_Evidence",
    "zip",
    root_dir=str(training_folder)
)

files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>